In [ ]:
import numpy as np
import os
import scipy
from load_data_function import load_data,save_data
import re
from load_data_function import fig_plot,battery_soh_plot,smooth_soh
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import gaussian_filter1d

In [ ]:
SNL_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\original_battery_data\SNL_dataset'
package_list=os.listdir(SNL_path)
#print(package_list)
SNL_data={}
SNL_SOH={}
for i,package in enumerate(package_list):

    print(f'package: {package}')
    battery_list=os.listdir(os.path.join(SNL_path,package))

    print(battery_list)

    package1={}
    package2={}
    for j,battery in enumerate(battery_list):
        print(f'battery: {battery}')
        battery_path=os.path.join(SNL_path,package,battery)
        battery_data = pd.read_csv(battery_path).dropna(subset=['Discharge_Capacity (Ah)', 'Charge_Capacity (Ah)'])

        #print(battery_data.shape)

        voltage=[]
        current=[]
        time=[]
        capacity=[]
        package1[f'battery_{j+1}']=[]
        package2[f'battery_{j+1}']=[]
        cycle_num= battery_data['Cycle_Index'].unique()
        cycle_num=sorted(cycle_num)
        #print(cycle_num)
        for k in cycle_num:
            cycle_data=battery_data[battery_data['Cycle_Index']==k]
            voltage=cycle_data['Voltage (V)'].values.reshape(1,-1)

            current=cycle_data['Current (A)'].values.reshape(1,-1)
            time_segment = cycle_data['Test_Time (s)'].values.reshape(1,-1)
            time=time_segment  # 转换为秒
            #time=time.reshape(1,-1)
            discharge_capacity=cycle_data['Discharge_Capacity (Ah)'].values.reshape(1,-1)
            charge_capacity=cycle_data['Charge_Capacity (Ah)'].values.reshape(1,-1)
            discharge_capacity=np.float32(discharge_capacity)
            #charge_capacity=np.float32(charge_capacity)
            #capacity=np.concatenate((discharge_capacity,charge_capacity),axis=1)
            if discharge_capacity.shape[1]<100:
                continue
            #print(discharge_capacity.shape)
            capacity_max=np.max(discharge_capacity)
            if package=='LFP':
                soh=capacity_max/1.1
            elif package=='NCA':
                soh=capacity_max/3.2
            else:
                soh=capacity_max/3
            #print(soh)
            package1[f'battery_{j+1}'].append(np.concatenate((voltage,current,time),axis=0))
            package2[f'battery_{j+1}'].append(soh)
    SNL_data[f'package_{i+1}']=package1
    SNL_SOH[f'package_{i+1}']=package2

In [ ]:
print(SNL_data.keys())
print(SNL_SOH.keys())
print(SNL_data['package_1'].keys())
print(SNL_SOH['package_1'].keys())


In [ ]:
package='package_3'
battery_soh_plot(SNL_SOH,SNL_SOH[package].keys(),package)

In [ ]:
fig_plot(SNL_SOH['package_1']['battery_5'])
print(SNL_SOH['package_1']['battery_5'])

In [ ]:
fig_plot(SNL_SOH['package_2']['battery_5'])

In [ ]:
fig_plot(SNL_SOH['package_3']['battery_5'])

In [ ]:
fig_plot(SNL_SOH['package_3']['battery_6'])

In [ ]:
print(SNL_SOH['package_1']['battery_1'])

In [ ]:

for package  in SNL_SOH.keys():
    for key in SNL_SOH[package].keys():
        #SNL_SOH[package][key]=gaussian_filter1d(SNL_SOH[package][key],sigma=1)
        for i in range(len(SNL_SOH[package][key])-1):
            if np.abs(SNL_SOH[package][key][i]-SNL_SOH[package][key][i+1])>0.1:
                SNL_SOH[package][key][i+1]= SNL_SOH[package][key][i]
        #SNL_SOH[package][key]=gaussian_filter1d(SNL_SOH[package][key],sigma=1)


In [ ]:
fig_plot(SNL_SOH['package_1']['battery_1'])

In [ ]:
smooth_SNL_SOH=smooth_soh(SNL_SOH,'gaussian',sigma=1)
package='package_1'
battery_soh_plot(smooth_SNL_SOH,smooth_SNL_SOH[package].keys(),package)

In [ ]:
fig_plot(smooth_SNL_SOH['package_1']['battery_3'])

In [ ]:
SNL_data=load_data('D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data\SNL_dataset/SNL_data.pkl')
SNL_SOH=load_data('D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data\SNL_dataset/SNL_SOH.pkl')

SNL_LFP_data={}
SNL_LFP_SOH={}
SNL_LFP_data['package_1']=SNL_data['package_1']
SNL_LFP_SOH['package_1']=smooth_SNL_SOH['package_1']

SNL_NCA_data={}
SNL_NCA_SOH={}
SNL_NCA_data['package_1']=SNL_data['package_2']
SNL_NCA_SOH['package_1']=smooth_SNL_SOH['package_2']

SNL_NMC_data={}
SNL_NMC_SOH={}
SNL_NMC_data['package_1']=SNL_data['package_3']
SNL_NMC_SOH['package_1']=smooth_SNL_SOH['package_3']

In [ ]:


save_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data\SNL_LFP_dataset'
os.makedirs(save_path,exist_ok=True)
save_data(SNL_LFP_data,os.path.join(save_path,'SNL_LFP_data.pkl'))
save_data(SNL_LFP_SOH,os.path.join(save_path,'SNL_LFP_SOH.pkl'))

save_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data\SNL_NCA_dataset'
os.makedirs(save_path,exist_ok=True)
save_data(SNL_NCA_data,os.path.join(save_path,'SNL_NCA_data.pkl'))
save_data(SNL_NCA_SOH,os.path.join(save_path,'SNL_NCA_SOH.pkl'))

save_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data\SNL_NMC_dataset'
os.makedirs(save_path,exist_ok=True)
save_data(SNL_NMC_data,os.path.join(save_path,'SNL_NMC_data.pkl'))
save_data(SNL_NMC_SOH,os.path.join(save_path,'SNL_NMC_SOH.pkl'))